In [1]:
import numpy as np
from scipy import constants
import xarray as xr
import uxarray as uxr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from scripts import umeshFcts as ufcts

from mpas_tools.cime.constants import constants
earthRadius = constants['SHR_CONST_REARTH']

# Create a global mesh for goSPL

Create an unstructured grid for a given cell width. The method relies on the UXarray and jigsaw libraries.

**In case where the mesh already exists it will not be recreated.**

Spherical mesh resolution km

| cell_width | edge_min  | edge_max | edge_mean | nodeNb |
| ---------- | ----------  | ---------- | ---------- | ---------- |
| 5 | 1.1 | 4.5 | 2.8 | 23632811 |
| 8 |  1.8 | 7.2 | 4.6  | 9236387 | 
| 10 | 2.2 | 8.9 | 5.7 | 5912778 |
| 15 | 3.3 | 13.1 | 8.6 | 2629742 |
| 20 | 4.5 | 18 | 11.5 | 1480168 |
| 25 | 5.6 | 22.4 | 14.4 | 947701 |
| 30 | 6.8 | 26.4 | 17.2 | 658525 |
| 35 | 8 | 30.5 | 20.1 |  484009 |

In [2]:
widthCell = 40
input_path = "input_"+str(widthCell) 

# Build the mesh
ufcts.buildGlobalMeshSimple(widthCell, input_path)

In [3]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc cellWidthVsLatLon.nc $input_path

mv: mesh.jig: No such file or directory
mv: mesh.log: No such file or directory
mv: mesh.msh: No such file or directory
mv: mesh-MESH.msh: No such file or directory
mv: mesh-HFUN.msh: No such file or directory
mv: mesh_triangles.nc: No such file or directory
mv: cellWidthVsLatLon.nc: No such file or directory


## Map variables on the UGRID 

We will now map global variables on this unstructured grid. In goSPL, typical variables would be:

- elevation (in m)
- vertical and horizontal tectonic forcing (displacement rates in m/yr)
- precipitation (in m/yr)
- dynamic topography (in m/yr)

Usually they will be provided in the form of `netcdf` or `geotiff` files. In both cases, the `xarray` or `rioxarray` libraries will allow you to open those files conveniently.

> Here we will use a netcdf grid containing all of these variables (except dynamic topography) for a give time interval.

In [4]:
# Loading the nc regular file
ncgrid = xr.open_dataset('data/251Ma.nc')
ncgrid

<xarray.Dataset> Size: 21MB
Dimensions:  (lat: 721, lon: 1441)
Coordinates:
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon      (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8 180.0
Data variables:
    h        (lat, lon) float64 8MB ...
    rain     (lat, lon) float32 4MB ...
    te       (lat, lon) float64 8MB ...

In case the file contains more variables than the ones you need for goSPL, you can select only the necessary ones:

In [5]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
ugrid = uxr.open_grid(ufile) 
# ugrid

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_251'
ufcts.inter2UGRID(ncgrid,ugrid,var_path,var_name,type='face')

In the `var_path` folder, you will find interpolated variables for the the UGRID (one file per variable) 

In [6]:
data_file = [var_path+'/'+var_name+'.nc']

# Get the information related to the mesh: primal and dual mesh
primal_mesh = uxr.open_dataset(ufile, *data_file, use_dual=False)
dual_mesh = uxr.open_dataset(ufile, *data_file, use_dual=True)

# Extract nodes and faces information
ucoords = np.empty((dual_mesh.uxgrid.n_node,3))
ucoords[:,0] = dual_mesh.uxgrid.node_x.values * earthRadius
ucoords[:,1] = dual_mesh.uxgrid.node_y.values * earthRadius
ucoords[:,2] = dual_mesh.uxgrid.node_z.values * earthRadius
ufaces = primal_mesh.uxgrid.node_face_connectivity.values

# Get information about your mesh:
print("Number of nodes: ",len(ucoords)," | number of faces ",len(ufaces))
edge_min = np.round(dual_mesh.uxgrid.edge_node_distances.min().values/1000. * earthRadius +0.,2)
edge_max = np.round(dual_mesh.uxgrid.edge_node_distances.max().values/1000.* earthRadius+0.,2)
edge_mean = np.round(dual_mesh.uxgrid.edge_node_distances.mean().values/1000.* earthRadius+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

Number of nodes:  370830  | number of faces  741656
edge range (km): min  9.25  | max  35.18  | mean  23.03


In [7]:
# Save voronoi mesh for visualisation purposes
saveVoro = False

if saveVoro:
    from mpas_tools.viz.paraview_extractor import extract_vtk
    extract_vtk(
            filename_pattern=ufile,
            variable_list='areaCell',
            dimension_list=['maxEdges=','nVertLevels=', 'nParticles='], 
            mesh_filename=ufile,
            out_dir=input_path, 
            ignore_time=True,
            # lonlat=True,
            xtime='none'
        )
    print("You could now visualise in Paraview (wireframe) the produced voronoi mesh!")
    print("This is a vtp mesh called: ", input_path+'/staticFieldsOnCells.vtp')

> You might want to check that everything went according to plan and look at the mesh and variables that will be used in goSPL.

To do so, we will build a `vtk` file that could be visualised in Paraview...

In [8]:
checkMesh = False

if checkMesh:
    import meshio

    paleovtk = input_path+"/init.vtk"

    vlist = list(primal_mesh.keys())
    vdata = []
    for k in vlist:
        vdata.append(primal_mesh[k].values)

    list_data = dict.fromkeys(el for el in vlist)
    list_data.update((k, vdata[i]) for i, k in enumerate(list_data))

    # Define mesh
    vis_mesh = meshio.Mesh(ucoords, {"triangle": ufaces}, 
                           point_data = list_data,
                        )
    # Write it disk
    meshio.write(paleovtk, vis_mesh)
    print("Writing VTK input file as {}".format(paleovtk))

Similar to what was done before we interpolate the structured variables to the UGRID mesh:

In [9]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
ugrid = uxr.open_grid(ufile) 
# ugrid

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_up_250'
ufcts.inter2UGRID(ncgrid,ugrid,var_path,var_name,type='face')

In [11]:
ugrid

<uxarray.Grid>
Original Grid Type: MPAS
Grid Dimensions:
  * n_node: 741656
  * n_edge: 1112484
  * n_face: 370830
  * n_max_face_nodes: 8
  * n_max_face_edges: 8
  * n_max_face_faces: 8
  * n_max_node_faces: 3
  * n_max_node_edges: 3
  * two: 2
Grid Coordinates (Spherical):
  * node_lon: (741656,)
  * node_lat: (741656,)
  * edge_lon: (1112484,)
  * edge_lat: (1112484,)
  * face_lon: (370830,)
  * face_lat: (370830,)
Grid Coordinates (Cartesian):
  * node_x: (741656,)
  * node_y: (741656,)
  * node_z: (741656,)
  * edge_x: (1112484,)
  * edge_y: (1112484,)
  * edge_z: (1112484,)
  * face_x: (370830,)
  * face_y: (370830,)
  * face_z: (370830,)
Grid Connectivity Variables:
  * face_edge_connectivity: (370830, 8)
  * node_edge_connectivity: (741656, 3)
  * node_face_connectivity: (741656, 3)
  * edge_node_connectivity: (1112484, 2)
  * face_face_connectivity: (370830, 8)
  * face_node_connectivity: (370830, 8)
  * edge_face_connectivity: (1112484, 2)
Grid Descriptor Variables:
  * face_areas: (370830,)
  * edge_face_distances: (1112484,)
  * edge_node_distances: (1112484,)
  * boundary_node_indices: (0,)

We then extract the tectonic forcing based on the displacement rates:

In [10]:
data_file = [var_path+'/'+var_name+'.nc']
dual_mesh = uxr.open_dataset(ufile, *data_file, use_dual=True)

> In case where you have specified a dynamic topography variable, your tectonic variable `tec` is a combination of the tectonic and dynamic topography components.

## goSPL input generation

We will now create the inputs for goSPL. We first start by creating the input mesh defining our UGRID structure:

In [14]:
ugrid.face_areas.max()

<xarray.DataArray 'face_areas' ()> Size: 8B
array(4.45339203e-05)
Attributes:
    long_name:  Area of each face.
    cf_role:    face_areas

array([1.40310452e+09, 1.17376997e+09, 1.37662681e+09, ...,
       1.15280270e+09, 1.12266095e+09, 1.37974354e+09], shape=(370830,))

In [21]:
meshname = var_path+"/mesh"
np.savez_compressed(meshname, v=ucoords, c=ufaces, 
                    z=dual_mesh.h.data
                    )

Now we save the forcing conditions (displacement rates, tectonic, precipitation...). Here you have the option to also add the next time step elevation, this will then be used in goSPL to force the model to match with the next paleo-elevation for specific regions (by defining the `zfit` parameter in the input file).

In [22]:
forcname = var_path+"/forcing251"

np.savez_compressed(forcname, 
                    te=dual_mesh.te.data, 
                    r=dual_mesh.rain.data,
                    )
